# Daphnet preprocessing: LOSO cross-validation with SMOTE
This notebook implements a clean preprocessing pipeline for the segmented Daphnet dataset.
- Loads the segmented CSV produced in the project outputs.
- Runs Leave-One-Subject-Out (LOSO) cross-validation.
- Within each LOSO training fold: scales features, then applies SMOTE to the training set only.
- Saves resampled training sets and untouched test sets for each fold to `outputs/daphnet_preprocessed/`.
The notebook handles both binary and multiclass labels.

In [ ]:
# If imbalanced-learn is not installed in the environment, uncomment and run the following line:
# !pip install -q imbalanced-learn

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Configuration: adjust paths or column names if necessary
DATA_PATH = Path('../../outputs/datasets_csv/daphnet_segmented_dataset.csv')
OUTPUT_DIR = Path('../../outputs/daphnet_preprocessed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# If you know the subject or label column names, set them here. Otherwise the notebook will try to infer them.
SUBJECT_COL = None  # e.g. 'subject' or 'participant'
LABEL_COL = None    # e.g. 'label' or 'class'

In [ ]:
# Load dataset (may be large)
df = pd.read_csv(DATA_PATH)
# Basic column discovery (one concise output)
print('Columns:', df.columns.tolist())

In [ ]:
def infer_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Try to infer subject and label columns if not provided
if SUBJECT_COL is None:
    SUBJECT_COL = infer_column(df, ['subject', 'Subject', 'subject_id', 'participant', 'pid', 'id'])
if LABEL_COL is None:
    LABEL_COL = infer_column(df, ['label', 'Label', 'class', 'y', 'target', 'FOG', 'freeze'])

if SUBJECT_COL is None or LABEL_COL is None:
    raise ValueError('Could not infer subject or label column. Please set SUBJECT_COL and LABEL_COL variables at the top of the notebook.')

print(f'Using subject column: {SUBJECT_COL}, label column: {LABEL_COL}')

In [ ]:
# Determine feature columns: drop non-feature columns like subject and label and any non-numeric columns
non_feature_cols = {SUBJECT_COL, LABEL_COL}
feature_cols = [c for c in df.columns if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df[c])]
if len(feature_cols) == 0:
    raise ValueError('No numeric feature columns found. Update feature selection logic or check your CSV.')

print(f'Found {len(feature_cols)} feature columns')

In [ ]:
def resample_fold(train_df, test_df, feature_cols, label_col, random_state=42):
    X_train = train_df[feature_cols].values
    y_train = train_df[label_col].values
    X_test = test_df[feature_cols].values
    y_test = test_df[label_col].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    sm = SMOTE(random_state=random_state)
    X_res, y_res = sm.fit_resample(X_train_scaled, y_train)

    # Return as DataFrames with the same column names for clarity
    X_res_df = pd.DataFrame(X_res, columns=feature_cols)
    y_res_df = pd.Series(y_res, name=label_col)
    X_test_df = pd.DataFrame(X_test_scaled, columns=feature_cols)
    y_test_df = pd.Series(y_test, name=label_col)

    return X_res_df, y_res_df, X_test_df, y_test_df, scaler

In [ ]:
# Main LOSO loop: for each subject, create train/test, apply SMOTE only on training set, and save outputs
subjects = df[SUBJECT_COL].unique()
summary = []
for subj in subjects:
    train_df = df[df[SUBJECT_COL] != subj].reset_index(drop=True)
    test_df = df[df[SUBJECT_COL] == subj].reset_index(drop=True)

    X_train_res, y_train_res, X_test_scaled, y_test, scaler = resample_fold(train_df, test_df, feature_cols, LABEL_COL)

    # Save per-fold datasets
    fold_prefix = OUTPUT_DIR / f'fold_subj_{subj}'
    fold_prefix.mkdir(parents=True, exist_ok=True)
    X_train_res.to_csv(fold_prefix / 'X_train_resampled.csv', index=False)
    y_train_res.to_csv(fold_prefix / 'y_train_resampled.csv', index=False)
    X_test_scaled.to_csv(fold_prefix / 'X_test_scaled.csv', index=False)
    y_test.to_csv(fold_prefix / 'y_test.csv', index=False)
    # Save scaler for potential downstream use
    # scaler can be saved with joblib if required (not saved by default here)

    summary.append({'subject': subj, 'train_samples_resampled': len(y_train_res), 'test_samples': len(y_test)})

# Minimal summary
summary_df = pd.DataFrame(summary)
print('Saved preprocessed folds to', OUTPUT_DIR)
print(summary_df.to_string(index=False))

In [ ]:
# Optional: plot label distribution before and after for the first fold (essential plot only)
import matplotlib.pyplot as plt
first_subj = subjects[0] if len(subjects)>0 else None
if first_subj is not None:
    # Load the saved files for the first fold to show distributions
    p = OUTPUT_DIR / f'fold_subj_{first_subj}'
    y_before = df[df[SUBJECT_COL] != first_subj][LABEL_COL]  # training before resample
    y_after = pd.read_csv(p / 'y_train_resampled.csv').squeeze()
    fig, axes = plt.subplots(1,2,figsize=(8,3))
    y_before.value_counts().plot.bar(ax=axes[0], title='Train before SMOTE')
    y_after.value_counts().plot.bar(ax=axes[1], title='Train after SMOTE')
    plt.tight_layout()
    plt.show()